# 🌌 APEX Launcher

Welcome to the **APEX** (Adaptive Platform for Unified AI Platform Configuration, Orchestration and Workspace Management) launcher. This notebook contains the **Stage-0 Bootstrap** loader which clones the GitHub repository, configures system search paths, installs dependencies, and boots the runtime dashboard.

### Stage-0 Bootstrap Sequence
1. **Environment Verification**: Detects if running on Google Colab or local systems.
2. **Google Drive Mount**: Prompts for Google Drive authentication in Colab if persistent storage is desired.
3. **Repository Clone**: Clones the codebase into target storage if missing.
4. **Path & Package Injection**: Updates `sys.path` to enable importing the runtime modules.
5. **Dependency Installation**: Runs pip installer requirements checks before loading application code.

In [ ]:
import sys
import subprocess
from pathlib import Path

# 1. Environment Detection & Storage Selection
is_colab = "google.colab" in sys.modules
repo_url = "https://github.com/Nikhil-Kumar-Shah/APEX.git"

# Default local target path
target_dir = Path.cwd().parent / "APEX"

if is_colab:
    print("[+] Google Colab environment detected.")
    # Mount Google Drive for persistent setup if desired
    mount_drive = input("Mount Google Drive for persistent storage? (y/n) [y]: ").strip().lower() != 'n'
    if mount_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            target_dir = Path("/content/drive/MyDrive/APEX")
        except Exception as e:
            print(f"[-] Google Drive mount failed: {e}. Falling back to temporary storage.")
            target_dir = Path("/content/APEX")
    else:
        target_dir = Path("/content/APEX")

print(f"[+] Target Installation Directory: {target_dir}")

# 2. Stage-0 Clone / Fetch
if not (target_dir / ".git").exists():
    print(f"[+] Repository not found. Cloning from {repo_url}...")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", repo_url, str(target_dir)], check=True)
else:
    print("[+] Existing repository detected. Fetching latest remote updates...")
    try:
        subprocess.run(["git", "-C", str(target_dir), "fetch", "--all"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"[-] Network fetch skipped (Offline Mode): {e}")

# 3. Update Python System Search Paths
repo_path_str = str(target_dir.resolve())
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)
else:
    sys.path.remove(repo_path_str)
    sys.path.insert(0, repo_path_str)

# 4. Install requirements dependencies
requirements_file = target_dir / "requirements.txt"
if requirements_file.is_file():
    print("[+] Installing Python dependencies from requirements.txt...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)], check=True)

# 5. Import and hand over to Stage-1 Bootstrap
try:
    from bootstrap.installer import InstallationWizard
    from bootstrap.launcher import RuntimeLauncher

    print("[+] Initializing Stage-1 Installation Wizard...")
    # Set interactive=False inside the notebook to auto-confirm if non-interactive run
    wizard = InstallationWizard(workspace_parent_dir=target_dir.parent, repo_url=repo_url)
    project_path = wizard.run(interactive=False)

    if project_path:
        launcher = RuntimeLauncher(project_path)
        launcher.launch()
    else:
        print("[-] Stage-1 Installation Wizard failed.")
except ImportError as e:
    print(f"[-] Failed to import bootstrap module: {e}. Check folder integrity.")